# Project 2

This project utilizes *openml* to read the dataset.

In [ ]:
import openml
import pandas as pd

# 1. Fetch the dataset using its ID
dataset = openml.datasets.get_dataset(31)

# 2. Get the data, specifying 'class' as the target variable
X, y, categorical_indicator, attribute_names = dataset.get_data(target="class")

df = pd.concat([X, y], axis=1)

## Data Preparation

The first step in perparing the data involes splitting the dataset into a test and training set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# Display the first few rows
print(X.head())

# Check for data types and missing values
print(X.info())

# Statistical summary of numerical features
print(X.describe())

This output shows an overview of the German Credit dataset. This contains 1,000 entries across 20 different attributes. It includes a mix of categorical and numerical data types. 

#### Checking for missing values.

In [ ]:
# Check for null values
print(X.isnull().sum())

This shows there is no missing values in the dataset. 

#### Understanding the target variable.

In [ ]:
# Check class distribution
print(y.value_counts(normalize=True))

This shows us that the dataset is imbalanced with 70% having good credit, and 30% having bad credit.

### Visualizing the Dataset

To help understand the data, we use boxplots and histograms to get a better representation of the content in the dataset. This allows us to check for outliers and get more insight into the distribuion inside the dataset. First, looking at the histograms:

In [ ]:
# Select numerical columns
num_cols = ['age', 'credit_amount', 'duration']

# Plot histograms
df[num_cols].hist(bins=20, figsize=(12, 8))
plt.suptitle('Distribution of Numerical Features')
plt.show()

The histograms illustrate that the **age** and **credit_amount** features exhibit a right-skewed distribution, showing that the majority of applicants are younger and request smaller loan amounts. Next, we examine the boxplots to check for outliers:

In [ ]:
# Boxplot for credit_amount
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['credit_amount'])
plt.title('Outlier Analysis: Credit Amount')
plt.show()

The boxplot reveals that the **credit_amount** feature is positively skewed, with a majority of loans concentrated at lower values.

Looking for correlations between attributes.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Encode categorical features temporarily for correlation analysis
df_encoded = df.copy()
for col in df_encoded.select_dtypes(['category', 'object']).columns:
    df_encoded[col] = df_encoded[col].cat.codes

# 2. Calculate the correlation matrix
corr_matrix = df_encoded.corr()

# 3. Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Attribute Correlation Heatmap')
plt.show()

### Preprocessing the Data

To prepare the dataset for machine learning we utilize a ColumnTransformer to standardize all the numerical features and apply one-hot encoding on all of the categorical features. This dual approach helps ensure that the features are correctly scaled and numerically represented for a consistent input for the model.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Identify feature types
numeric_features = ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
categorical_features = ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']

# 2. Define the transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

# 3. Fit and transform the data (using the training set)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## Model Selection